# 30 — Baselines de prevalencia y heurística

Establece referencias reproducibles antes de entrenar modelos. Este notebook **no consume TEST**. Evalúa únicamente sobre el split de validación congelado por M3.

In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m3_gate = paths.docs / "decisions" / "M3_GATE.md"
assert m3_gate.exists() and "M3 ✅ CLOSED" in m3_gate.read_text(encoding="utf-8"), (
    "M3 no está formalmente cerrado. No iniciar Modeling."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
MODEL_DIR = paths.root / "artifacts" / "models" / "m4"
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

{
    "gold_schema": GOLD_SPEC.schema_version,
    "modeling_schema": MODEL_SPEC.schema_version,
    "target": MODEL_SPEC.target,
    "model_matrix": MODEL_SPEC.model_matrix,
    "operational_review_fraction": MODEL_SPEC.ranking.operational_fraction,
}

from argos.models.baselines import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


In [ ]:
from argos.models.baselines import build_baselines

result = build_baselines(
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
)
result.metrics


In [ ]:
result.ranking

In [ ]:
result.validation

In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
print("30 ✅ PASS" if result.ready_for_next_notebook else "30 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook
